# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

print("Setup complete")

Setup complete


In [15]:
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    hf_token = getpass.getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute("SET VARIABLE hf_token = ?", [hf_token])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Warehouse connected")
print("Feature month: February 2026")
print("Outcome month: March 2026")

Enter your Hugging Face READ token: ··········
Warehouse connected
Feature month: February 2026
Outcome month: March 2026


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I chose **Logistic Regression** because this lane is a ranking problem where the goal is to identify client-content pairs that are more likely to receive clicks in the following month.

Logistic Regression is a good first learned model because it is simple, fast, interpretable, and provides a probability that can be used for ranking.

The model uses only February signals that were available at the decision moment. The March outcome is used only as the target.

I will compare the learned model with the Week-4 baseline using the same held-out client groups and the same ranking metrics.

In [16]:
FEATURES = [
    "gsc_clicks",
    "gsc_impressions",
    "avg_position",
]

TARGET = "target"

print("Features:", FEATURES)
print("Target:", TARGET)
print("Model rows:", len(model_frame))

Features: ['gsc_clicks', 'gsc_impressions', 'avg_position']
Target: target
Model rows: 303572


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a **time-aware split**.

- Feature window: February 2026
- Decision moment: February 28, 2026
- Outcome window: March 2026
- Group: client-content pair

February data is used to construct the features, while March clicks are used only as the outcome.

I will also keep client groups together when splitting the modeling rows so that the same client does not appear in both training and validation sets.

This avoids using future March information to construct February features and gives a more honest estimate of performance on unseen client groups.

In [17]:
# Grouped holdout split by client.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    gss.split(
        model_frame,
        model_frame[TARGET],
        groups=model_frame["client_hash_id"],
    )
)

train_df = model_frame.iloc[train_idx].copy()
test_df = model_frame.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Train clients:",
    train_df["client_hash_id"].nunique()
)

print(
    "Test clients:",
    test_df["client_hash_id"].nunique()
)

print(
    "Client overlap:",
    len(
        set(train_df["client_hash_id"])
        & set(test_df["client_hash_id"])
    )
)

Train rows: 264134
Test rows: 39438
Train clients: 40
Test clients: 10
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
    ),
])

model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print("Model trained successfully")

Model trained successfully


In [19]:
BASELINE_PATH = "/content/sample_data/baseline_action_score.csv"

baseline_df = pd.read_csv(BASELINE_PATH)

print("Baseline rows:", len(baseline_df))
print("Baseline columns:")
print(baseline_df.columns.tolist())

Baseline rows: 142079
Baseline columns:
['client_hash_id', 'content_hash_id', 'gsc_clicks', 'gsc_impressions', 'avg_position', 'ctr', 'volume_score', 'ctr_gap', 'action_score', 'reason_code', 'action_label']


In [21]:
# Align the baseline with the modeling data using the shared client-content keys.

comparison_df = model_frame.merge(
    baseline_df[
        [
            "client_hash_id",
            "content_hash_id",
            "action_score",
            "reason_code",
            "action_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Comparison rows:", len(comparison_df))
print("Comparison columns:")
print(comparison_df.columns.tolist())

comparison_df.head()

Comparison rows: 134035
Comparison columns:
['client_hash_id', 'content_hash_id', 'gsc_clicks', 'gsc_impressions', 'avg_position', 'gsc_clicks_mar', 'target', 'action_score', 'reason_code', 'action_label']


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,avg_position,gsc_clicks_mar,target,action_score,reason_code,action_label
0,client_e547b89c05043229,content_1eea820697c3b95a,0.0,299.0,12.946228,0.0,0,0.000000,CTR_OPPORTUNITY,REVIEW_CTR
1,client_e547b89c05043229,content_5f58c55cbfee172a,0.0,514.0,10.490023,0.0,0,0.000000,CTR_OPPORTUNITY,REVIEW_CTR
2,client_e547b89c05043229,content_6fe390ba3af1e456,3.0,2931.0,38.436254,5.0,1,-0.008171,CTR_OPPORTUNITY,REVIEW_CTR
3,client_e547b89c05043229,content_3ad5d2160242b9ca,2.0,970.0,9.710810,1.0,1,-0.014182,CTR_OPPORTUNITY,REVIEW_CTR
4,client_e547b89c05043229,content_a2bd730a7cf68316,1.0,551.0,6.017373,1.0,1,-0.011458,CTR_OPPORTUNITY,REVIEW_CTR


In [22]:
# Build the held-out test set with model scores.
test_df = model_frame.loc[X_test.index].copy()

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print("Test rows:", len(test_df))
test_df.head()

Test rows: 39438


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,avg_position,gsc_clicks_mar,target,model_score
0,client_e547b89c05043229,content_1eea820697c3b95a,0.0,299.0,12.946228,0.0,0,0.122889
1,client_e547b89c05043229,content_9abd8b303f805847,6.0,733.0,6.495085,4.0,1,0.984817
2,client_e547b89c05043229,content_5f58c55cbfee172a,0.0,514.0,10.490023,0.0,0,0.170201
3,client_e547b89c05043229,content_6fe390ba3af1e456,3.0,2931.0,38.436254,5.0,1,0.993133
4,client_e547b89c05043229,content_3ad5d2160242b9ca,2.0,970.0,9.710810,1.0,1,0.726642


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
# Error analysis on the held-out test set

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
import pandas as pd

# Predicted class using the same 0.5 threshold used for classification
test_df["predicted"] = (test_df["model_score"] >= 0.5).astype(int)

# Identify error types
test_df["error_type"] = "Correct"

test_df.loc[
    (test_df["target"] == 0) & (test_df["predicted"] == 1),
    "error_type"
] = "False Positive"

test_df.loc[
    (test_df["target"] == 1) & (test_df["predicted"] == 0),
    "error_type"
] = "False Negative"

# Basic error counts
error_counts = test_df["error_type"].value_counts()

print("Error summary:")
print(error_counts)

print("\nClassification report:")
print(
    classification_report(
        test_df["target"],
        test_df["predicted"],
        digits=3
    )
)

# Show a small sample of model mistakes
errors = test_df[test_df["error_type"] != "Correct"][
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_clicks",
        "gsc_impressions",
        "avg_position",
        "gsc_clicks_mar",
        "target",
        "model_score",
        "error_type"
    ]
]

print("\nSample model errors:")
display(errors.head(10))

# ---------------------------------------------------------
# Feature interpretation using permutation importance
# ---------------------------------------------------------

feature_names = X_test.columns.tolist()

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc"
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print("\nPermutation importance:")
display(importance_df)

Error summary:
error_type
Correct           34402
False Negative     4344
False Positive      692
Name: count, dtype: int64

Classification report:
              precision    recall  f1-score   support

           0      0.863     0.975     0.916     28080
           1      0.910     0.618     0.736     11358

    accuracy                          0.872     39438
   macro avg      0.887     0.796     0.826     39438
weighted avg      0.877     0.872     0.864     39438


Sample model errors:


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,avg_position,gsc_clicks_mar,target,model_score,error_type
5,client_e547b89c05043229,content_a2bd730a7cf68316,1.0,551.0,6.017373,1.0,1,0.355215,False Negative
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,0.0,291.0,4.333115,1.0,1,0.128301,False Negative
11,client_e547b89c05043229,content_65f27fbd014ef1c7,1.0,526.0,7.492398,1.0,1,0.343173,False Negative
15,client_e547b89c05043229,content_e87512f3582b175c,3.0,676.0,14.104353,0.0,0,0.792786,False Positive
16,client_e547b89c05043229,content_4170f9c4f674077a,0.0,273.0,4.925861,1.0,1,0.124467,False Negative
17,client_e547b89c05043229,content_f0969d4f2e6e7719,1.0,543.0,2.878582,1.0,1,0.357364,False Negative
21,client_e547b89c05043229,content_1c38606759cb3b6f,1.0,324.0,8.334021,1.0,1,0.269612,False Negative
25,client_e547b89c05043229,content_182074b4379a3089,0.0,386.0,9.801025,1.0,1,0.142408,False Negative
30,client_e547b89c05043229,content_8e6c48f786cc8ad2,1.0,861.0,6.194193,1.0,1,0.481571,False Negative
31,client_e547b89c05043229,content_ce7d9f679a308649,0.0,746.0,7.767651,4.0,1,0.236421,False Negative



Permutation importance:


,feature,importance
1,gsc_impressions,0.154321
0,gsc_clicks,0.124537
2,avg_position,-0.002230


### Error interpretation

The model was evaluated on the held-out test set rather than the training data.

The errors are mainly of two types:

- **False positives:** pages predicted as likely to receive a March click but which did not receive one.
- **False negatives:** pages that received at least one March click but were assigned a probability below the 0.5 decision threshold.

This shows that the model is useful for ranking and decision support, but its predictions are not perfect. Low-volume pages can be difficult to classify because small changes in clicks can change the binary outcome.

Permutation importance was used to inspect which February features the model relies on most. The importance values are directional: they show which available features contribute most to held-out ROC-AUC performance, not causal effects.

The model should therefore be treated as a ranking/decision-support tool rather than an automatic decision maker.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.